# Сравнение HAS

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns

import json
import math
import os
import glob
import re

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ========== НАСТРОЙКИ ==========
EXPERIMENTS_ROOT = Path('../dataset-generator/experiments/6')
METHOD_CV = 'p90-p50-window-200-lb-smooth-div2'
STABILIZATION_TICK = 500
MEAN_REQUESTS = 44.3
MEAN_DURATION = 7.0
ALGORITHMS = ['LAS', 'RR', 'GREEDY_OPT', 'HAS']
CVAR_QUANTILE = 0.95
# ================================

In [ ]:
def get_lognorm_sigma_rq_in_task(data):
    dist = data['distribution']
    rq_in_task = dist['requests-in-task-amount']

    return rq_in_task['params']['sigma']

def create_point(meta_json_path: Path):
    with open(meta_json_path, 'r', encoding='utf-8') as file:
        meta_json_data = json.load(file)
        th_load =  round((MEAN_REQUESTS * MEAN_DURATION) / meta_json_data["arrival-interval"], 2)
        
        sigma_lognorm = get_lognorm_sigma_rq_in_task(meta_json_data)
        th_cv = round(math.sqrt(math.exp(sigma_lognorm**2) - 1), 2)
        
        return {"th_cv": th_cv, "th_load": th_load}

# -> {"FCFS": {"cvar": 1000, "emp_load": 10, "emp_cv": 1}, "LAS": {...}, ...} 
def process_run(scenario_run: Path):
    run = {}
    
    min_cvar = float("inf")
    for algo in ALGORITHMS:
        result = {}
        algo_files = glob.glob(f"{algo}*.csv", root_dir=scenario_run)
        for algo_file in algo_files:
            if algo_file == f'{algo}.csv':
                cvar = compute_cvar(scenario_run.joinpath(algo_file), CVAR_QUANTILE)
                result['cvar'] = cvar
                min_cvar = min(min_cvar, cvar)

            if re.fullmatch(fr"{algo}.*_cv\.csv", algo_file):
                result["emp_cv"] = compute_avg_value(scenario_run.joinpath(algo_file))

            if algo_file == f'{algo}_load.csv':
                result["emp_load"] = compute_avg_value(scenario_run.joinpath(algo_file))
        run[algo] = result

    for algo in ALGORITHMS:
        algo_cvar = run[algo]['cvar']
        run[algo]['deviation'] = (algo_cvar - min_cvar) / (min_cvar) * 100
    
    return run

def process_scenario(scenario_path: Path):
    point = create_point(scenario_path.joinpath('meta.json'))

    algos = {}
    for run_dir in scenario_path.iterdir():
        if run_dir.is_dir():
            run = process_run(run_dir)
            for key, value in run.items():
                algos[key] = merge_result(algos.get(key, {}), value)

    point['algos'] = algos
    return point

def merge_result(old_value, new_value):
    if len(old_value) == 0:
        merged = {}
        for key, value in new_value.items():
            merged[key] = [value]
        return merged
        
    for key in old_value.keys():
        old_value[key].append(new_value[key])

    return old_value 

def compute_cvar(file: Path, quantile=CVAR_QUANTILE):
    data = pd.read_csv(file)
    percentile = data['duration_sec'].quantile(quantile)
    above = data[data['duration_sec'] > percentile]
    avg_above = above['duration_sec'].mean()

    return float(avg_above)

def compute_avg_value(file: Path, stabilization_tick=STABILIZATION_TICK):
    df = pd.read_csv(file)
    df = df[df['tick'] > stabilization_tick]
    return df['value'].dropna().mean().item()

def create_points(experiments_root: Path, verbose=False):
    points = []
    for experiment in experiments_root.iterdir():
        if experiment.is_file():
            continue
        for scenario in experiment.iterdir():
            if scenario.is_file():
                continue
            points.append(process_scenario(scenario))
            if verbose:
                print(f"Обработана папка {scenario}")
    return points



In [ ]:
def get_points(path: Path, use_cache=True, verbose=False):
    points_file = path.joinpath("points.json")
    file_not_exists = not points_file.exists()
    if not use_cache or file_not_exists:
        print(f"Создание {points_file}")
        points = create_points(path, verbose)
        with open(points_file, "w") as f:
            json.dump(points, f)
        return points

    print(f"Загрузка из {points_file}")
    with open(points_file, 'r', encoding='utf-8') as file:
        return json.load(file)

# кэшируем после первого прогона
points = get_points(Path(EXPERIMENTS_ROOT), use_cache=True)

In [ ]:
# возвращает среднее значение метрики по каждому алгоритму
def get_mean(point, key):
    return { 
        algo: sum(values[key]) / len(values[key])
        for algo, values in point["algos"].items()
    }

def get_mean_cvars(point):
    return get_mean(point, "cvar")

def get_mean_deviation(point):
    return get_mean(point, "deviation")
    
def get_winner(point):
    mean_cvars = get_mean_cvars(point)
    return min(mean_cvars, key=mean_cvars.get)

In [ ]:
rows = []

for point in points:
    mean_cvar = get_mean_deviation(point)
    rows.append({"th_cv": point["th_cv"], "th_load": point["th_load"]} | mean_cvar)

df = pd.DataFrame(rows)
df = df.rename(columns={"GREEDY_OPT": "FCFS"})
df

In [ ]:
ALGORITHMS = ['HAS', 'LAS', 'RR', 'FCFS']
ALGO_NAMES = {'HAS': 'HAS', 'LAS': 'LAS', 'RR': 'RR', 'FCFS': 'FCFS'}

# %%
# Строим матрицы для всех алгоритмов
matrices = {}
for algo in ALGORITHMS:
    pivot = df.pivot_table(
        index='th_cv', 
        columns='th_load', 
        values=algo,
        aggfunc='first'
    )
    pivot = pivot.sort_index(ascending=False)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    matrices[algo] = pivot
    print(f"Матрица для {ALGO_NAMES[algo]}: {pivot.shape[0]} × {pivot.shape[1]}")

In [ ]:
# %% [markdown]
# # Сравнение алгоритмов: матрицы преимущества (совмещённые тепловые карты)
# 
# Для каждого алгоритма (HAS, LAS, RR, FCFS) вычисляется отклонение от лучшего в каждой ячейке.
# Положительное число = алгоритм хуже лучшего (проигрыш), отрицательное = алгоритм лучше лучшего (выигрыш).

all_values = []
for pivot in matrices.values():
    all_values.extend(pivot.values.flatten())
all_values = [v for v in all_values if not np.isnan(v)]

vmin = min(all_values)
vmax = max(all_values)
# Ограничиваем для лучшей читаемости (опционально)
vmin_display = max(vmin, -50)  # показываем не ниже -50%
vmax_display = min(vmax, 150)  # показываем не выше 150%

print(f"Диапазон отклонений: [{vmin:.1f}%, {vmax:.1f}%]")
print(f"Отображаемый диапазон: [{vmin_display:.1f}%, {vmax_display:.1f}%]")

# %%
fig2, axes2 = plt.subplots(1, 4, figsize=(20, 12))
axes2 = axes2.flatten()

for idx, (algo, pivot) in enumerate(matrices.items()):
    ax = axes2[idx]
    
    plot_data = pivot.clip(vmin_display, vmax_display)
    annot_data = pivot.round(1)
    
    sns.heatmap(plot_data, annot=annot_data, fmt='.1f', cmap='RdYlGn_r',
                center=0, vmin=vmin_display, vmax=vmax_display,
                cbar=False,
                linewidths=0.5, square=True,
                annot_kws={'fontsize': 12}, ax=ax)
    
    cv_labels = [f"{cv:.2f}" for cv in pivot.index]
    load_labels = [f"{load:.2f}" for load in pivot.columns]
    
    ax.set_yticklabels(cv_labels, fontsize=7)
    ax.set_xticklabels(load_labels, rotation=45, ha='right', fontsize=7)
    
    ax.set_xlabel('ρ', fontsize=9)
    ax.set_ylabel('CV', fontsize=9)
    ax.set_title(f'{ALGO_NAMES[algo]}', fontsize=20)

fig2.suptitle('', fontsize=14)
plt.tight_layout()
plt.savefig(EXPERIMENTS_ROOT / 'all_algorithms_advantage_horizontal.png', dpi=150, bbox_inches='tight')
plt.show()
